# Treino do modelo de visão — FlyHub

Fecha o ciclo: dataset anotado no Roboflow → `models/best.pt` e
`models/metrics.json`.

**Você edita uma célula só**, a de parâmetros logo abaixo. O resto roda de
cima para baixo. Ao terminar, copie os dois arquivos para `models/` (a última
célula faz isso) e confira na tela **Voo** que o badge mudou. Não há mais
nenhum passo na aplicação — ela detecta o arquivo novo sozinha.

Instalação: `pip install -r notebooks/requirements.txt` (~2,5 GB, só nesta
máquina). Detalhes e a armadilha do rebalanceamento: `notebooks/README.md`.


## 1. Parâmetros

A única célula que se edita. `ROBOFLOW_API_KEY` sai do ambiente de
propósito — chave colada aqui vira chave commitada.


In [ ]:
import os
from pathlib import Path

# --- Roboflow -------------------------------------------------------------
ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', '')
ROBOFLOW_WORKSPACE = 'seu-workspace'
ROBOFLOW_PROJECT = 'seu-projeto'
ROBOFLOW_VERSION = 1
ROBOFLOW_FORMAT = 'yolov11'   # yolov8 tem o mesmo layout

# --- Treino ---------------------------------------------------------------
BASE_MODEL = 'yolo11n.pt'     # yolo11s/m/l para modelos maiores
EPOCHS = 100
IMGSZ = 640
BATCH = 16
DEVICE = None                 # None = automático; 'cpu', '0', 'cuda'

# --- Conferência da partição ---------------------------------------------
# O Roboflow reparticiona ao gerar uma versão. Ligado, o notebook compara
# com o split temporal da coleta e avisa. Ver notebooks/README.md.
CHECK_SPLIT = True
STRICT_SPLIT = False          # True aborta o treino em vez de só avisar
SPLIT_MANIFEST = None         # None = o split_manifest.json mais recente

# --- Caminhos -------------------------------------------------------------
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MODELS_DIR = REPO / 'models'          # onde a aplicação lê
DATASETS_DIR = REPO / 'data' / 'datasets'   # coletas, com os manifestos
DOWNLOAD_DIR = Path('datasets') / f'v{ROBOFLOW_VERSION}'

print('repositório :', REPO)
print('modelos     :', MODELS_DIR)
print('chave       :', 'definida' if ROBOFLOW_API_KEY else 'AUSENTE — exporte ROBOFLOW_API_KEY')


## 2. Baixar o dataset anotado

Pule esta célula se o dataset já está em disco — basta apontar
`DATA_YAML` para o `data.yaml` dele na célula seguinte.

> **Ao gerar a versão no Roboflow, escolha *Keep existing split*.**
> *Rebalance* redistribui as imagens aleatoriamente e desfaz o split
> temporal — silenciosamente. A célula 3 confere.


In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = project.version(ROBOFLOW_VERSION).download(
    ROBOFLOW_FORMAT, location=str(DOWNLOAD_DIR)
)

DATA_YAML = Path(dataset.location) / 'data.yaml'
print('data.yaml:', DATA_YAML)


## 3. Conferir a partição

Compara as proporções do dataset baixado com as do `split_manifest.json` da
coleta. Divergência aqui significa que o Roboflow rebalanceou: quadros
vizinhos no tempo voltaram a cair em partições diferentes, o modelo
memoriza, e a métrica de validação sobe para um número que não se sustenta
em voo novo.

Portado de `train/train.py` do M4TD.


In [ ]:
import json, re

SPLIT_KEYS = (('train', 'train'), ('val', 'valid'), ('test', 'test'))
TOLERANCE_PP = 5.0   # desvio em pontos percentuais a partir do qual é suspeito
IMAGE_SUFFIXES = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')


def read_yaml(path: Path) -> dict:
    import yaml  # vem com o ultralytics

    return yaml.safe_load(path.read_text(encoding='utf-8')) or {}


def split_dir(data_yaml: Path, config: dict, key: str):
    """Resolve `train:`/`val:`/`test:` do data.yaml para uma pasta de imagens."""
    value = config.get(key)
    if not isinstance(value, str) or not value:
        return None
    base = Path(str(config.get('path') or data_yaml.parent))
    if not base.is_absolute():
        base = (data_yaml.parent / base).resolve()
    candidate = Path(value)
    if not candidate.is_absolute():
        candidate = (base / candidate).resolve()
    # O Roboflow escreve `train/images`; alguns exports apontam para a pasta pai.
    if candidate.is_dir() and (candidate / 'images').is_dir():
        candidate = candidate / 'images'
    return candidate if candidate.is_dir() else None


def count_images(directory) -> int:
    if directory is None:
        return 0
    return sum(1 for e in directory.iterdir()
               if e.is_file() and e.suffix.lower() in IMAGE_SUFFIXES)


def latest_manifest():
    """O split_manifest.json da versão mais recente em data/datasets/."""
    found = []
    if not DATASETS_DIR.is_dir():
        return None
    for entry in DATASETS_DIR.iterdir():
        match = re.match(r'^v(\d+)\.(\d)$', entry.name)
        if match and entry.is_dir() and (entry / 'split_manifest.json').is_file():
            found.append(((int(match.group(1)), int(match.group(2))), entry))
    if not found:
        return None
    _, newest = max(found)
    return newest.name, json.loads((newest / 'split_manifest.json').read_text('utf-8'))


def check_split(data_yaml: Path) -> dict:
    config = read_yaml(data_yaml)
    downloaded = {s: count_images(split_dir(data_yaml, config, k)) for k, s in SPLIT_KEYS}
    total = sum(downloaded.values())
    result = {
        'downloaded': downloaded,
        'downloaded_proportions': {
            s: round(n / total * 100, 1) if total else None for s, n in downloaded.items()
        },
        'manifest_version': None,
        'manifest': None,
        'warnings': [],
        'ok': True,
    }

    if not downloaded['test']:
        result['warnings'].append(
            'o data.yaml não aponta para nenhuma imagem de test — não haverá conjunto de teste'
        )

    source = (
        (Path(SPLIT_MANIFEST).parent.name, json.loads(Path(SPLIT_MANIFEST).read_text('utf-8')))
        if SPLIT_MANIFEST
        else latest_manifest()
    )
    if source is None:
        result['warnings'].append(
            'nenhum split_manifest.json em data/datasets/ para comparar — '
            'a partição do dataset baixado não pôde ser conferida'
        )
        result['ok'] = not result['warnings']
        return result

    version, manifest = source
    counts = {s: int((manifest.get('counts') or {}).get(s) or 0) for _, s in SPLIT_KEYS}
    local_total = sum(counts.values())
    result['manifest_version'] = version
    result['manifest'] = {
        'counts': counts,
        'proportions': {
            s: round(n / local_total * 100, 1) if local_total else None
            for s, n in counts.items()
        },
    }

    for split in counts:
        want = result['manifest']['proportions'][split]
        got = result['downloaded_proportions'][split]
        if want is None or got is None:
            continue
        if abs(got - want) > TOLERANCE_PP:
            result['warnings'].append(
                f'{split}: o dataset baixado tem {got}% das imagens e o split temporal '
                f'de {version} tem {want}% — a partição não é a mesma'
            )

    result['ok'] = not result['warnings']
    return result


def print_check(check: dict) -> None:
    print('\n--- partição do dataset ---')
    header = f"{'':8} {'baixado':>18}"
    if check['manifest']:
        header += f"  split temporal {check['manifest_version']}"
    print(header)
    for _, split in SPLIT_KEYS:
        got, gp = check['downloaded'][split], check['downloaded_proportions'][split]
        line = f"{split:8} {got:>8} {f'({gp}%)' if gp is not None else '':>9}"
        if check['manifest']:
            want = check['manifest']['counts'][split]
            wp = check['manifest']['proportions'][split]
            line += f"  {want:>10} {f'({wp}%)' if wp is not None else '':>9}"
        print(line)
    if check['ok']:
        print('\n  OK — a partição bate com o split temporal.')
        return
    print('\n  ATENÇÃO')
    for warning in check['warnings']:
        print(f'    - {warning}')
    print(
        '\n  O Roboflow reparticiona ao gerar uma versão. Se ela foi gerada com'
        '\n  rebalanceamento, quadros vizinhos no tempo voltaram a cair em partições'
        '\n  diferentes: o modelo memoriza e a métrica não se sustenta em voo novo.'
        '\n  Ver notebooks/README.md.'
    )


check = {'downloaded': {}, 'downloaded_proportions': {}, 'manifest': None,
         'manifest_version': None, 'warnings': [], 'ok': True}
if CHECK_SPLIT:
    check = check_split(DATA_YAML)
    print_check(check)
    if STRICT_SPLIT and not check['ok']:
        raise SystemExit('abortado por STRICT_SPLIT')


## 4. Treinar


In [ ]:
from ultralytics import YOLO
import time

RUN_NAME = f"flyhub-{time.strftime('%Y%m%d-%H%M%S')}"

model = YOLO(BASE_MODEL)
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name=RUN_NAME,
    device=DEVICE,
)

RUN_DIR = Path(getattr(model.trainer, 'save_dir', Path('runs/detect') / RUN_NAME)).resolve()
BEST = RUN_DIR / 'weights' / 'best.pt'
assert BEST.is_file(), f'{BEST} não foi produzido pelo treino'
print('pesos do run:', BEST)


## 5. Validar e montar o `metrics.json`

A extração é tolerante de propósito: a forma de `results.box` mudou entre
versões do Ultralytics, e um atributo ausente vira `null` no JSON em vez de
derrubar o treino inteiro depois de horas de GPU.


In [ ]:
import hashlib

results = model.val(data=str(DATA_YAML), imgsz=IMGSZ, device=DEVICE)


def number(value):
    try:
        out = float(value)
    except (TypeError, ValueError):
        return None
    return None if out != out else round(out, 5)   # descarta NaN


def sequence(value):
    try:
        return list(value) if value is not None else []
    except TypeError:
        return []


def sha256(path: Path):
    """Identidade do arquivo de pesos.

    O mtime muda a cada cópia; o hash não. É ele que permite à aplicação
    dizer que o metrics.json é de um treino diferente do best.pt carregado.
    """
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()


box = getattr(results, 'box', None)
names = dict(getattr(model, 'names', {}) or {})
indices = [int(i) for i in sequence(getattr(box, 'ap_class_index', None))]
ap50, maps = sequence(getattr(box, 'ap50', None)), sequence(getattr(box, 'maps', None))
precision, recall = sequence(getattr(box, 'p', None)), sequence(getattr(box, 'r', None))

per_class = [
    {
        'class_id': class_id,
        'name': str(names.get(class_id, class_id)),
        'map50': number(ap50[position]) if position < len(ap50) else None,
        # `maps` é indexado por id de classe, não pela posição na lista
        'map50_95': number(maps[class_id]) if class_id < len(maps) else None,
        'precision': number(precision[position]) if position < len(precision) else None,
        'recall': number(recall[position]) if position < len(recall) else None,
    }
    for position, class_id in enumerate(indices)
]

document = {
    'generated_at': time.time(),
    'generated_at_iso': time.strftime('%Y-%m-%dT%H:%M:%S%z'),
    'source': 'notebooks/treino-yolo.ipynb',
    'weights': {
        'path': 'models/best.pt',
        'sha256': sha256(BEST),
        'size_bytes': BEST.stat().st_size,
        'from_run': str(BEST),
    },
    'training': {
        'base_model': BASE_MODEL,
        'epochs': EPOCHS,
        'imgsz': IMGSZ,
        'batch': BATCH,
        'name': RUN_NAME,
        'run_dir': str(RUN_DIR),
        'device': DEVICE,
    },
    'dataset': {
        'data_yaml': str(DATA_YAML),
        'name': DATA_YAML.parent.name,
        'counts': check['downloaded'],
        'proportions': check['downloaded_proportions'],
        'split_manifest_version': check['manifest_version'],
        'split_check_ok': check['ok'],
        'split_warnings': check['warnings'],
    },
    'metrics': {
        'map50': number(getattr(box, 'map50', None)),
        'map50_95': number(getattr(box, 'map', None)),
        'precision': number(getattr(box, 'mp', None)),
        'recall': number(getattr(box, 'mr', None)),
        'fitness': number(getattr(results, 'fitness', None)),
    },
    'per_class': per_class,
    'classes': [str(names[k]) for k in sorted(names)] if names else [],
}
document['metrics']


## 6. Entregar

Copia para `models/`. **É o último passo do treino e o primeiro da**
**aplicação** — daqui em diante nada mais precisa ser feito: o backend
confere o `mtime` do arquivo no máximo uma vez por segundo e recarrega
sozinho.

A escrita do JSON é em arquivo temporário seguido de `replace`, que é
atômico: sem isso a aplicação poderia ler o `metrics.json` pela metade
no instante da cópia.


In [ ]:
import shutil

MODELS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(BEST, MODELS_DIR / 'best.pt')

temporary = MODELS_DIR / 'metrics.json.tmp'
temporary.write_text(json.dumps(document, ensure_ascii=False, indent=2), encoding='utf-8')
temporary.replace(MODELS_DIR / 'metrics.json')

print(f"""
--- pronto ---
  pesos     {MODELS_DIR / 'best.pt'}
  métricas  {MODELS_DIR / 'metrics.json'}
  run       {RUN_DIR}

  mAP@50     {document['metrics']['map50']}
  mAP@50-95  {document['metrics']['map50_95']}
  precision  {document['metrics']['precision']}
  recall     {document['metrics']['recall']}

Agora abra a tela Voo. Em segundos o badge sobre o vídeo deve mudar de
'SEM MODELO — vídeo cru' para 'MODELO best.pt'. Não reinicie nada.

Depois: commit e push DO NOTEBOOK. Os pesos não vão para o Git.
""")
if not check['ok']:
    print('  Lembrete: a partição divergiu do split temporal — as métricas',
          'acima podem estar otimistas.')


## O que fazer se o badge não mudar

1. O arquivo está em `models/best.pt` na **raiz do repositório**?
   `ls -l models/`
2. O container enxerga? `docker compose exec backend ls -l /models`
3. `GET /api/v1/model` diz o quê? O campo `message` explica os quatro
   estados possíveis.
4. A inferência pode estar **desligada** — o badge diz
   `MODELO DESLIGADO — vídeo cru`. É o toggle da tela Voo, e ele sobrevive a
   reinício de propósito.
5. Arquivo reescrito com o mesmo `mtime` (raro): botão **Recarregar**.

Passo a passo completo em [`docs/modelo/index.md`](../docs/modelo/index.md).
